In [1]:
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import random
from PIL import ImageFont

from MonsterNameGenerator import MarkovMonsterNameGenerator
from textGenerateUtils import (generate_scientific_name, generate_prompt, generate_description,
                              refine_prompt_with_image, generate_profile, pick_traits)
from imageGenerateUtils import get_image, add_caption


In [2]:
# model setup

# 画像モデルは ComfyUI コンテナ側 (./models/comfyui/) に配置する。
# 切り替えたい場合は imageGenerateUtils の DIFFUSION_MODEL / COMFYUI_DIFFUSION_MODEL 環境変数で指定する。

monster_name_filepath = 'monsterNames.txt'

title_font = ImageFont.truetype("ipagp.ttf", 27)
paragraph_font = ImageFont.truetype("ipagp.ttf", 15)
caption_font = ImageFont.truetype("ipagp.ttf", 12)
# 学名は斜体で組む。ipagp にイタリック体が無いので Noto Serif Italic を併用する
italic_font = ImageFont.truetype("NotoSerif-Italic.ttf", 12)


In [3]:
# ComfyUI サーバーの疎通と、必要なモデルが配置されているかを確認する
import json, urllib.request
from imageGenerateUtils import COMFYUI_URL, DIFFUSION_MODEL, TEXT_ENCODER, VAE

def _choices(node, field):
    info = json.load(urllib.request.urlopen(f"{COMFYUI_URL}/object_info/{node}"))
    return info[node]["input"]["required"][field][0]

print("ComfyUI:", COMFYUI_URL)
for node, field, want in [("UNETLoader", "unet_name", DIFFUSION_MODEL),
                          ("CLIPLoader", "clip_name", TEXT_ENCODER),
                          ("VAELoader", "vae_name", VAE)]:
    found = _choices(node, field)
    print(f"  {node:<12} {want:<32} {'OK' if want in found else 'NOT FOUND -> ' + str(found)}")
    assert want in found, f"{want} が ComfyUI 側に見つかりません"


ComfyUI: http://comfyui:8188
  UNETLoader   novaAnimeAM_v40.safetensors      OK
  CLIPLoader   qwen_3_06b_base.safetensors      OK
  VAELoader    qwen_image_vae.safetensors       OK


In [4]:
name_generator = MarkovMonsterNameGenerator(n=2)
name_generator.train_from_file(monster_name_filepath)

In [5]:
fields = [
    "杉林", "古代林", "畑", "草むら", "花畑", "密林", "水没林","ジャングル","峠","山の麓","樹海","竹林","森","霧の森","熱帯雨林","サバンナ","桜並木","果樹園",
    "洞窟", "鍾乳洞","谷底","岩石地帯","鉱山","荒野","岩の中",
    "雪原","凍土","氷河",
    "旧市街地", "化学工場跡地","都市の下水道", "古城","都市部","廃工場","地下鉄廃線","空中都市",
    "大砂漠", "オアシス",
    "海", "深海", "浅瀬", "砂浜", "汽水域", "川底","孤島","海底遺跡","湖","潮溜まり","地下水路","滝","沈没船","サンゴ礁",
    "成層圏","惑星中心部","溶岩地帯",
    "モンスターの体内"
]
spicies = [
    "生物",
    "鳥",
    "虫",
    "植物",
    "花",
    "草",
    "木",
    "キノコ",
    "魚",
    "爬虫類",
    "哺乳類",
    "両生類",
    "巨大生物",
    "小型生物",
    "草食動物",
    "肉食動物",
    "寄生生物",
    "絶滅危惧種",
    "甲殻類",
    "貝",
    "群生生物",
    "原始生物",
    "人工生命",
    "分類不明の生物"
]

In [ ]:
finalImages = []
for j in tqdm(range(25)):
    name = name_generator.generate()
    field = random.choice(fields)
    if field == "モンスターの体内":
        field = name_generator.generate()+"の体内"
    spicy = random.choice(spicies)
    if spicy in ("貝","草","鳥","魚") and random.randint(0,1)==1:
        name = name + spicy
    target = "{0}にて観測される架空の{1}「{2}」".format(field, spicy, name)
    print(target)
    # 危険度・個体数・構図はコード側で振る。カードには出ない裏設定
    traits = pick_traits()
    print("{0} / {1} / {2}".format(traits["composition"]["label"], traits["danger"], traits["population"]))
    description = generate_description(target)
    scientific_name = generate_scientific_name(target, description)
    # 説明文の後に作る。説明文と矛盾しない範囲で、説明文に無い見た目の細部を足す役
    profile = generate_profile(target, description, traits)
    prompt = generate_prompt(target, description, profile, traits)
    extra_negative = traits["composition"]["negative"]
    background_draft = get_image(prompt.strip(), extra_negative=extra_negative)
    refined_prompt = refine_prompt_with_image(prompt.strip(), background_draft, target, description, profile, traits)
    background = get_image(refined_prompt.strip(), extra_negative=extra_negative)

    finalImage = add_caption(name, description, scientific_name, background, title_font, paragraph_font, caption_font, italic_font)

    finalImage.save("endemic/{0}-{1}.png".format(j,name))
    finalImages.append(finalImage)

  0%|          | 0/25 [00:00<?, ?it/s]

潮溜まりにて観測される架空の爬虫類「モレクトリス」
生息地の風景 / 毒を持ち、接触すると危険 / 記録が数例しかない希少種
潮溜まりに生息する小型の爬虫類である。甲殻類や小魚を捕食するため、口元には鋭い歯を持つ。体色は周囲の砂利や藻と混ざりやすく、発見されにくい。夜行性であり、昼間は岩陰などで隠れていることが多い。
Litorinodactylus cryptochromus
体長: 全長15センチメートル前後の小型爬虫類。
体色と質感: 湿った砂利や藻と同化した緑褐色で、表面は粘液を帯びて滑らかである。
頭部: 扁平した頭頂に小さな眼が位置し、口元には鋭い鋳歯状の歯列を持つ。
付属肢: 脚4本（片側2本）、短く太く吸盤状の趾を持つ。
特徴的な器官: 腹部側に毒液を蓄える小型の腺があり、皮膚から微かに臭気を発する。
食性: 甲殻類や小魚を捕食し、鋭い歯で噛み砕いて食べる。
人間への危険度: 毒を持ち、接触すると危険。
個体数: 記録が数例しかない希少種。
行動と姿勢: 夜行性であり、昼間は岩陰に身を潜め静止している。
生息環境の細部: 潮溜まりの浅瀬、湿った砂利地帯、藻類が絡まる岩陰付近。
shallow tidal pool, wet pebble bed, tangled algae around rocks, damp stone shadows, small reptile partially concealed under rock cover, flattened head with small eyes, sharp serrated teeth visible in mouth, greenish-brown mottled skin resembling wet gravel and algae, smooth slimy surface texture, four short thick legs with suction cup toes, two legs on each side, no other limbs, one specimen shown clearly, whole body visible, resting motionless in daytime shelter, faint scent emanating from 

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(40,24))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img in tqdm(zip(axes.flatten(), finalImages)):
    ax.imshow(img)
    ax.axis('off')